## Lab 3 — Prompt Engineering

In [97]:
import os
from dotenv import load_dotenv
from groq import Groq

In [98]:
load_dotenv()

True

In [99]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [100]:
MODEL = "openai/gpt-oss-120b"

def generate_response(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

## Exercise 1: Doctor Summary Generator

Generate a structured doctor's summary from patient consultation notes.

In [101]:
doctor_prompt = """
Create a doctor's summary using these patient details.

Patient: {patient_name}
Notes: {patient_notes}

Use this format:

Symptoms:
Diagnosis:
Recommendation:

Use only the provided information.
Write "Not specified" for missing information.
Return only the summary.
"""

In [102]:
patient_name = "Rahul Sharma"

patient_notes = """
Fever and headache for two days.
A viral infection is suspected.
Advised rest, fluids, and follow-up if symptoms worsen.
"""

In [103]:
prompt = doctor_prompt.format(
    patient_name=patient_name,
    patient_notes=patient_notes
)

print(generate_response(prompt))

Symptoms: Fever and headache for two days.  
Diagnosis: A viral infection is suspected.  
Recommendation: Advised rest, fluids, and follow-up if symptoms worsen.


## Exercise 2: Review Classifier

In [104]:
review_prompt = """
Classify the following customer review into exactly one category.

POSITIVE: The customer is satisfied.
NEGATIVE: The customer is dissatisfied.
NEUTRAL: The review expresses no clear positive or negative opinion.
MIXED: The review contains both positive and negative opinions.
UNRELATED: The text is not a meaningful product or service review.

Review: {review_text}

Return only the category name in uppercase.
"""

In [105]:
review_text = "The headphones have excellent sound quality, but the battery drains too quickly."

prompt = review_prompt.format(review_text=review_text)

print(generate_response(prompt).strip())

MIXED


## Exercise 3: Resume Field Extractor

In [106]:
resume_prompt = """
Extract only the requested fields from the resume.

Resume:
{resume_text}

Fields to extract:
{fields_to_extract}

Return valid JSON only.
Include exactly the requested fields.
Use null if a field is missing.
Do not invent information.
"""

In [107]:
import json

resume_text = """
Name: Vedant Patil
Email: vedant.patil@example.com
Phone: 9876543210
Skills: Python, SQL, AWS
Education: B.Tech in Computer Science
"""

fields_to_extract = ["name", "email", "skills"]

In [108]:
prompt = resume_prompt.format(
    resume_text=resume_text,
    fields_to_extract=json.dumps(fields_to_extract)
)

response = generate_response(prompt)
print(response)

{
  "name": "Vedant Patil",
  "email": "vedant.patil@example.com",
  "skills": ["Python", "SQL", "AWS"]
}


## Exercise 4: Customer Support Reply Generator

In [109]:
support_prompt = """
You are a professional customer support representative for {company_name}.

Customer message:
{customer_message}

Write a polite, empathetic, and helpful response.
Address the customer's concern and suggest a practical next step.
Keep the response within {max_words} words.
Do not invent company policies or make promises.
Return only the reply.
"""

In [110]:
customer_message = """
I ordered a laptop five days ago, but it has not arrived.
The tracking page has not been updated. Please help.
"""

company_name = "TechStore"
max_words = 60

In [111]:
prompt = support_prompt.format(
    company_name=company_name,
    customer_message=customer_message,
    max_words=max_words
)

reply = generate_response(prompt)
print(reply)

print("\nWord count:", len(reply.split()))

Hi, I’m sorry to hear your laptop hasn’t arrived and the tracking isn’t updating. Could you please share your order number? I’ll investigate the shipment and contact the carrier for you. We’ll keep you posted on the status as soon as possible. Thank you for your patience.

Word count: 47


## Exercise 5: Multi-language Translator

In [112]:
translation_prompt = """
Translate the following sentence into {target_language}.

Sentence:
{sentence}

Formality: {formality}

Instructions:
- Preserve the original meaning.
- Use natural, grammatically correct language.
- Follow the requested level of formality.
- Return only the translated sentence.
"""

In [113]:
sentence = "Could you please help me resolve this issue?"
target_language = "Hindi"
formality = "Formal"

In [114]:
prompt = translation_prompt.format(
    sentence=sentence,
    target_language=target_language,
    formality=formality
)

print(generate_response(prompt))

क्या आप कृपया इस समस्या को हल करने में मेरी सहायता कर सकते हैं?


## Exercise 6: Iterative Content Refiner

In [115]:
draft_text = """
Our application help citizen to submit complaints and track there status.
It make the grievance process easy and fast.
"""

issue_to_fix = "grammar, clarity, and readability"

In [116]:
draft = draft_text

for round_number in range(1, 4):
    prompt = f"""
Improve the following draft by fixing: {issue_to_fix}.

Round: {round_number}

Draft:
{draft}

Instructions:
- Preserve the original meaning.
- Correct grammar and improve readability.
- Return only the revised text.
"""

    draft = generate_response(prompt)

    print(f"Round {round_number}:")
    print(draft)
    print("-" * 50)

Round 1:
Our application helps citizens submit complaints and track their status. It makes the grievance process easy and fast.
--------------------------------------------------
Round 2:
Our application enables citizens to submit complaints and track their status, making the grievance process simple and swift.
--------------------------------------------------
Round 3:
Our application allows citizens to submit complaints and track their status, streamlining the grievance process.
--------------------------------------------------
